In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
divyanshyecho_enso_ham2019_dataset_path = kagglehub.dataset_download('divyanshyecho/enso-ham2019-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install netCDF4 torch-geometric scipy -q
import netCDF4, torch_geometric
from scipy.stats import pearsonr
print('netCDF4:', netCDF4.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.5 MB/s eta 0:00:00
netCDF4: 1.7.4
torch_geometric: 2.7.0
OK


In [ ]:
import os, gc, random
import numpy as np
import netCDF4 as nc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BASE = '/kaggle/input/datasets/divyanshyecho/enso-ham2019-dataset'

CKPT_DIR       = '/kaggle/working/cp5_leads_4_7'
LEADS_TO_TRAIN = [4, 5, 6, 7]
# ─────────────────────────────────────────────────────────────

os.makedirs(CKPT_DIR, exist_ok=True)

CMIP5_INPUT     = f'{BASE}/CMIP5.input.36mn.1861_2001.nc'
SODA_INPUT      = f'{BASE}/SODA.input.36mn.1871_1970.nc'
GODAS_INPUT     = f'{BASE}/GODAS.input.36mn.1980_2015.nc'
CMIP5_LABEL_3MV = f'{BASE}/CMIP5.label.nino34.12mn_3mv.1863_2003.nc'
CMIP5_LABEL_2MV = f'{BASE}/CMIP5.label.nino34.12mn_2mv.1863_2003.nc'
SODA_LABEL_3MV  = f'{BASE}/SODA.label.nino34.12mn_3mv.1873_1972.nc'
SODA_LABEL_2MV  = f'{BASE}/SODA.label.nino34.12mn_2mv.1873_1972.nc'
GODAS_LABEL_3MV = f'{BASE}/GODAS.label.12mn_3mv.1982_2017.nc'
GODAS_LABEL_2MV = f'{BASE}/GODAS.label.12mn_2mv.1982_2017.nc'

N_NODES    = 1393
IN_FEATS   = 72
BATCH_SIZE = 32
EPOCHS     = 300
SEED       = 42

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f'CKPT_DIR:       {CKPT_DIR}')
print(f'LEADS_TO_TRAIN: {LEADS_TO_TRAIN}')
print(f'EPOCHS:         {EPOCHS}')
print('Config ready.')

Device: cuda
CKPT_DIR:       /kaggle/working/cp5_leads_4_7
LEADS_TO_TRAIN: [4, 5, 6, 7]
EPOCHS:         300
Config ready.


In [ ]:
ds_soda  = nc.Dataset(SODA_INPUT)
sst_all  = np.array(ds_soda.variables['sst'][:, 0, :, :])
t300_all = np.array(ds_soda.variables['t300'][:, 0, :, :])
lats     = np.array(ds_soda.variables['lat'][:])
lons     = np.array(ds_soda.variables['lon'][:])
ds_soda.close()

land_mask  = (sst_all  == 0.0).all(axis=0) & \
             (t300_all == 0.0).all(axis=0)
ocean_mask = ~land_mask
ocean_idx  = np.where(ocean_mask.flatten())[0]
N_NODES    = int(ocean_mask.sum())

lat_grid = np.repeat(lats[:, None], 72, axis=1).flatten()[ocean_idx]
lon_grid = np.repeat(lons[None, :], 24, axis=0).flatten()[ocean_idx]

print(f'Ocean nodes: {N_NODES}')
print(f'Land  nodes: {24*72 - N_NODES}')

Ocean nodes: 1393
Land  nodes: 335


In [ ]:
def load_input(input_file, sst_var='sst'):
    ds   = nc.Dataset(input_file)
    sst  = np.array(ds.variables[sst_var][:]).astype(np.float32)
    t300 = np.array(ds.variables['t300'][:]).astype(np.float32)
    ds.close()
    X = np.stack([sst, t300], axis=1)
    X = np.nan_to_num(X, nan=0.0)
    mean = X.mean(axis=(0, 2), keepdims=True)
    std  = X.std( axis=(0, 2), keepdims=True) + 1e-6
    X    = (X - mean) / std
    N    = X.shape[0]
    X    = X.reshape(N, 2, 36, -1)[:, :, :, ocean_idx]
    X    = X.transpose(0, 3, 1, 2).reshape(N, N_NODES, -1)
    return X

def load_labels(label_file):
    ds = nc.Dataset(label_file)
    pr = np.array(ds.variables['pr'][:]).astype(np.float32)
    ds.close()
    return pr

def get_lead_data(lead):
    """
    Correct forecast pairing: Input[i] → Labels[i+1]
    lead=0  → n=1  → 2mv label file
    lead=1+ → n=2+ → 3mv label file
    """
    if lead == 0:
        cmip5_lbl = CMIP5_LABEL_2MV
        soda_lbl  = SODA_LABEL_2MV
        godas_lbl = GODAS_LABEL_2MV
    else:
        cmip5_lbl = CMIP5_LABEL_3MV
        soda_lbl  = SODA_LABEL_3MV
        godas_lbl = GODAS_LABEL_3MV

    X_cmip5 = load_input(CMIP5_INPUT, sst_var='sst1')
    X_soda  = load_input(SODA_INPUT,  sst_var='sst')
    X_godas = load_input(GODAS_INPUT, sst_var='sst')

    L_cmip5 = load_labels(cmip5_lbl)
    L_soda  = load_labels(soda_lbl)
    L_godas = load_labels(godas_lbl)

    # THE FIX: Input[i] → Labels[i+1]
    X_cmip5 = X_cmip5[:-1];  Y_cmip5 = L_cmip5[1:, lead, 0, 0]
    X_soda  = X_soda[:-1];   Y_soda  = L_soda[1:,  lead, 0, 0]
    X_godas = X_godas[:-1];  Y_godas = L_godas[1:, lead, 0, 0]

    X_train = np.concatenate([X_cmip5, X_soda], axis=0)
    Y_train = np.concatenate([Y_cmip5, Y_soda], axis=0)

    return X_train, Y_train, X_godas, Y_godas

print('Data loading functions ready.')

Data loading functions ready.


In [ ]:
ds_s = nc.Dataset(SODA_INPUT)
sr   = np.array(ds_s.variables['sst'][:]).astype(np.float32)
tr   = np.array(ds_s.variables['t300'][:]).astype(np.float32)
ds_s.close()

_X = np.stack([sr, tr], axis=1)
_X = np.nan_to_num(_X, nan=0.0)
_X = _X.reshape(100, 2, 36, -1)[:, :, :, ocean_idx]
_X = _X.transpose(0, 3, 1, 2).reshape(100, N_NODES, -1)

soda_sst_mean  = _X[:, :, :36].mean(axis=(0, 2))
soda_t300_mean = _X[:, :, 36:].mean(axis=(0, 2))
lat_norm = (lat_grid - lat_grid.mean()) / (lat_grid.std() + 1e-6)
lon_norm = (lon_grid - lon_grid.mean()) / (lon_grid.std() + 1e-6)

static_np = np.stack([soda_sst_mean, soda_t300_mean,
                      lat_norm, lon_norm], axis=1)
X_static  = torch.tensor(static_np, dtype=torch.float32).to(device)

print(f'Static features: {X_static.shape}')
del _X, sr, tr
gc.collect()

Static features: torch.Size([1393, 4])


512

In [ ]:
class StructureLearner(nn.Module):
    def __init__(self, static_feats=4, hidden=16, a1=0.1, a2=2.0):
        super().__init__()
        self.a1   = a1
        self.a2   = a2
        self.lin1 = nn.Linear(static_feats, hidden, bias=False)
        self.lin2 = nn.Linear(static_feats, hidden, bias=False)

    def forward(self, X_static):
        M1 = torch.tanh(self.a1 * self.lin1(X_static))
        M2 = torch.tanh(self.a1 * self.lin2(X_static))
        A  = torch.sigmoid(self.a2 * (M1 @ M2.T))
        A  = A / (A.sum(dim=1, keepdim=True) + 1e-6)
        A  = 0.5 * A + 0.5 * torch.eye(N_NODES, device=A.device)
        return A

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W  = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, A, X):
        out = torch.stack([A @ X[i] for i in range(X.size(0))], dim=0)
        out = self.W(out)
        B, N, D = out.shape
        out = self.bn(out.reshape(B*N, D)).reshape(B, N, D)
        return F.elu(out)

class Graphino(nn.Module):
    def __init__(self, in_feats=72, hidden=250, static_feats=4):
        super().__init__()
        self.structure = StructureLearner(static_feats=static_feats)
        self.gcn1      = GCNLayer(in_feats, hidden)
        self.gcn2      = GCNLayer(hidden,   hidden)
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Linear(128, 1)
        )

    def forward(self, X, X_static):
        A  = self.structure(X_static)
        Z1 = self.gcn1(A, X)
        Z2 = self.gcn2(A, Z1) + Z1
        Z  = torch.cat([Z1, Z2], dim=-1)
        g  = Z.mean(dim=1)
        return self.mlp(g).squeeze(-1)

set_seed(SEED)
_m = Graphino().to(device)
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
_x = torch.randn(4, N_NODES, IN_FEATS).to(device)
with torch.no_grad():
    _o = _m(_x, X_static)
print(f'Output shape: {_o.shape}')
del _m, _x, _o
gc.collect()
print('Architecture OK')

Parameters: 146,141
Output shape: torch.Size([4])
Architecture OK


In [ ]:
def train_lead(lead):
    print(f'\n{"="*55}')
    print(f'Training lead={lead}  →  n={lead+1} months ahead')
    print(f'{"="*55}')

    pred_file   = f'{CKPT_DIR}/preds_lead{lead:02d}_seed{SEED}.npy'
    latest_path = f'{CKPT_DIR}/latest_lead{lead:02d}_seed{SEED}.pt'
    best_path   = f'{CKPT_DIR}/best_lead{lead:02d}_seed{SEED}.pt'

    # Skip if already fully trained
    if os.path.exists(pred_file):
        print(f'  Already done — loading saved predictions.')
        _, _, _, Y_test = get_lead_data(lead)
        preds = np.load(pred_file)
        cc    = pearsonr(preds, Y_test)[0]
        print(f'  CC = {cc:.4f}')
        return preds, Y_test

    # Load data
    X_train, Y_train, X_test, Y_test = get_lead_data(lead)
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    Y_tr = torch.tensor(Y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tr, Y_tr),
        batch_size=BATCH_SIZE, shuffle=True)

    # Fresh model for each lead
    set_seed(SEED)
    model = Graphino(in_feats=IN_FEATS).to(device)
    opt   = torch.optim.Adam(
                model.parameters(),
                lr=0.0005,
                weight_decay=1e-6)

    # ── Cosine annealing LR schedule ─────────────────────────
    # Smooth decay from 0.0005 to 1e-5 over all EPOCHS
    # No plateau detection — no premature LR drops
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt,
                T_max=EPOCHS,
                eta_min=1e-5)

    best_cc     = -999.0
    start_epoch = 1

    # Resume from checkpoint if exists
    if os.path.exists(latest_path):
        print(f'  Resuming from checkpoint...')
        ckpt = torch.load(latest_path, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        opt.load_state_dict(ckpt['opt_state'])
        sched.load_state_dict(ckpt['sched_state'])
        best_cc     = ckpt['best_cc']
        start_epoch = ckpt['epoch'] + 1
        print(f'  Resumed at epoch {start_epoch} | '
              f'best CC = {best_cc:.4f}')
    else:
        print(f'  Starting fresh.')

    print(f'  Training for {EPOCHS} epochs | cosine LR schedule')
    print(f'  {"-"*50}')

    for epoch in range(start_epoch, EPOCHS + 1):

        # Train
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb, X_static)
            loss = F.mse_loss(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)

        # Step LR scheduler every epoch
        sched.step()

        # Evaluate on GODAS
        model.eval()
        preds_te = []
        with torch.no_grad():
            for i in range(0, len(X_te), BATCH_SIZE):
                xb = X_te[i:i+BATCH_SIZE].to(device)
                preds_te.append(
                    model(xb, X_static).cpu().numpy())
        preds_te = np.concatenate(preds_te)
        cc = pearsonr(preds_te, Y_test)[0]

        # Save best checkpoint
        if cc > best_cc:
            best_cc = cc
            torch.save(model.state_dict(), best_path)

        # Save latest every epoch — crash protection
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'opt_state'   : opt.state_dict(),
            'sched_state' : sched.state_dict(),
            'best_cc'     : best_cc,
        }, latest_path)

        if epoch % 10 == 0 or epoch == 1:
            lr = opt.param_groups[0]['lr']
            print(f'  Epoch {epoch:3d} | loss={avg_loss:.4f} | '
                  f'CC={cc:.4f} | best={best_cc:.4f} | '
                  f'lr={lr:.6f}')

    # Load best weights and save predictions
    model.load_state_dict(torch.load(
        best_path, map_location=device, weights_only=False))
    model.eval()
    preds_best = []
    with torch.no_grad():
        for i in range(0, len(X_te), BATCH_SIZE):
            xb = X_te[i:i+BATCH_SIZE].to(device)
            preds_best.append(
                model(xb, X_static).cpu().numpy())
    preds_best = np.concatenate(preds_best)

    np.save(pred_file, preds_best)

    if os.path.exists(latest_path):
        os.remove(latest_path)

    print(f'\n  DONE. Best CC = {best_cc:.4f}')

    del model, X_tr, Y_tr, X_te, X_train, Y_train, X_test
    gc.collect()
    torch.cuda.empty_cache()

    return preds_best, Y_test

print('train_lead() ready.')

train_lead() ready.


In [ ]:
all_preds  = []
all_labels = []

for lead in [4, 5, 6, 7]:
    preds, labels = train_lead(lead)
    all_preds.append(preds)
    all_labels.append(labels)
    cc = pearsonr(preds, labels)[0]
    print(f'\nLead {lead:2d} (n={lead+1:2d}) finished: CC = {cc:.4f}')
    print('='*55)

print('\nAccount 2 complete — leads 4,5,6,7 (n=5 to n=8)')
print(f'Download .npy files from {CKPT_DIR}')


Training lead=4  →  n=5 months ahead
  Starting fresh.
  Training for 300 epochs | cosine LR schedule
  --------------------------------------------------
  Epoch   1 | loss=0.3996 | CC=-0.0354 | best=-0.0354 | lr=0.000500
  Epoch  10 | loss=0.3551 | CC=0.1609 | best=0.1972 | lr=0.000499
  Epoch  20 | loss=0.3554 | CC=0.1585 | best=0.2039 | lr=0.000495
  Epoch  30 | loss=0.3436 | CC=0.2340 | best=0.2340 | lr=0.000488
  Epoch  40 | loss=0.3341 | CC=0.1745 | best=0.2541 | lr=0.000479
  Epoch  50 | loss=0.3355 | CC=0.2086 | best=0.2541 | lr=0.000467
  Epoch  60 | loss=0.3279 | CC=0.1930 | best=0.2541 | lr=0.000453
  Epoch  70 | loss=0.3183 | CC=0.1439 | best=0.2541 | lr=0.000437
  Epoch  80 | loss=0.3090 | CC=0.2153 | best=0.2541 | lr=0.000419
  Epoch  90 | loss=0.2948 | CC=0.2015 | best=0.2587 | lr=0.000399
  Epoch 100 | loss=0.2903 | CC=0.2047 | best=0.2587 | lr=0.000377
  Epoch 110 | loss=0.2756 | CC=0.1669 | best=0.2919 | lr=0.000355
  Epoch 120 | loss=0.2660 | CC=0.1991 | best=0.303

In [ ]:
import numpy as np

SEED = 42
CKPT_DIR = '/kaggle/working/cp5_leads_4_7'

for lead in [4, 5, 6, 7]:
    path = f'{CKPT_DIR}/preds_lead{lead:02d}_seed{SEED}.npy'
    preds = np.load(path)
    print(f'\nlead{lead:02d}_preds = np.array({preds.tolist()})')


lead04_preds = np.array([0.10617165267467499, -0.6077020168304443, -0.6117839813232422, -0.16022437810897827, 0.32054147124290466, -0.3247717022895813, -0.8282560110092163, 0.6359168291091919, 0.34924206137657166, -0.21927444636821747, -0.04222532734274864, 0.1902996152639389, -0.2828178405761719, -0.42872917652130127, -0.46161797642707825, 0.2008771002292633, -0.9295137524604797, -0.06411506980657578, -0.06271962821483612, -0.13826392590999603, -0.16226688027381897, 0.33854249119758606, 0.6754249930381775, 0.4146404266357422, 0.6542019248008728, -0.6453995108604431, 0.1554020345211029, 0.10525894165039062, -0.8042078614234924, -0.2289271354675293, -0.0918898731470108, -0.3586651384830475, 0.8344433903694153, 0.8280954360961914, -0.3516499400138855])

lead05_preds = np.array([-0.32461798191070557, -0.46575310826301575, -0.08807288110256195, -0.2605612277984619, 0.5007767081260681, -0.27099040150642395, -0.8310720920562744, 0.9051726460456848, 0.44091200828552246, 0.19329389929771423, 

In [ ]:
from scipy.stats import pearsonr
import numpy as np

CKPT_DIR = '/kaggle/working/cp5_leads_4_7'
SEED = 42

for lead in [4, 5, 6, 7]:
    preds  = np.load(f'{CKPT_DIR}/preds_lead{lead:02d}_seed{SEED}.npy')
    _, _, _, Y_test = get_lead_data(lead)
    cc = pearsonr(preds, Y_test)[0]
    print(f'Lead {lead:2d} (n={lead+1:2d}): CC = {cc:.4f}')

Lead  4 (n= 5): CC = 0.4196
Lead  5 (n= 6): CC = 0.3706
Lead  6 (n= 7): CC = 0.4578
Lead  7 (n= 8): CC = 0.5272
